In [ ]:
# define different matrics

import numpy as np
import pandas as pd
from scipy import stats
from typing import Optional, Tuple
import sys

EPS = 1e-8



def calc_kge(obs: np.ndarray, pred: np.ndarray) -> float:
    
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() < 2: # 
        return np.nan
    obs, pred = obs[mask], pred[mask]

   
    r_mat = np.corrcoef(obs, pred)
    r = r_mat[0, 1]
    
    
    mean_obs = np.mean(obs)
    beta = np.mean(pred) / (mean_obs + EPS)
    
   
    std_obs = np.std(obs)
    alpha = np.std(pred) / (std_obs + EPS)
    
    if std_obs == 0:
       
        return np.nan 

    
    kge = 1.0 - np.sqrt((r - 1.0)**2 + (alpha - 1.0)**2 + (beta - 1.0)**2)
    return float(kge)


def calc_nse(obs: np.ndarray, pred: np.ndarray) -> float:
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() < 2:
        return np.nan
    obs, pred = obs[mask], pred[mask]

    numerator = np.sum((obs - pred) ** 2)
    
    denominator = np.sum((obs - np.mean(obs)) ** 2)
    
    if denominator == 0:
        
        return np.nan 
        
    nse = 1.0 - (numerator / (denominator + EPS))
    return float(nse)


def calc_pbias_peak(obs: np.ndarray, pred: np.ndarray) -> float:
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() == 0:
        return np.nan
    obs, pred = obs[mask], pred[mask]
    
    peak_obs = np.max(obs)
    peak_pred = np.max(pred)
    
    pbias = 100.0 * (peak_pred - peak_obs) / (peak_obs + EPS)
    return float(pbias)


def calc_pbias_volume(obs: np.ndarray, pred: np.ndarray) -> float:
   
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() == 0:
        return np.nan
    obs, pred = obs[mask], pred[mask]
    
    vol_obs = np.sum(obs)
    vol_pred = np.sum(pred)
    
    pbias = 100.0 * (vol_pred - vol_obs) / (vol_obs + EPS)
    return float(pbias)


def calc_r_shape(obs: np.ndarray, pred: np.ndarray) -> float:
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() < 2:
        return np.nan
    obs, pred = obs[mask], pred[mask]
    
   
    if np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan 
        
    r_mat = np.corrcoef(obs, pred)
    return float(r_mat[0, 1])


def calc_peak_timing_error(obs: np.ndarray, pred: np.ndarray) -> float:
    
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() == 0:
        return np.nan
    obs, pred = obs[mask], pred[mask]
    
    if len(obs) == 0:
         return np.nan
         
    tpeak_obs = np.argmax(obs)
    tpeak_pred = np.argmax(pred)
    
    return float(tpeak_pred - tpeak_obs)


def calc_frac_peak_in_20pct(obs: np.ndarray, pred: np.ndarray) -> float:
    
    pbias = calc_pbias_peak(obs, pred)
    if pd.isna(pbias):
        return np.nan
    
    if -20.0 <= pbias <= 20.0:
        return 1.0
    else:
        return 0.0



def calc_crps(obs: np.ndarray, pred_ens: np.ndarray) -> float:
   
    pred_ens_t = pred_ens.T 
    
    daily_crps_scores = []
    
    for t in range(len(obs)):
        obs_t = obs[t]
        ens_t = pred_ens_t[t]
        
       
        if pd.isna(obs_t) or np.isnan(ens_t).any():
            continue 
        
        
        ens_t_sorted = np.sort(ens_t)
        M = len(ens_t_sorted)
        if M == 0:
            continue
            
        
        term1 = np.mean(np.abs(ens_t_sorted - obs_t))
       
        term2 = np.sum(ens_t_sorted * (2 * np.arange(1, M + 1) - M - 1))
        term2 /= (M**2)
        
        daily_crps = term1 - term2
        daily_crps_scores.append(daily_crps)

    if not daily_crps_scores:
        return np.nan
   
    return float(np.mean(daily_crps_scores))


def calc_iqr_mean(pred_ens: np.ndarray) -> float:
    
    if np.isnan(pred_ens).all():
        return np.nan
        
    
    q75 = np.nanpercentile(pred_ens, 75, axis=0) # [7,]
    q25 = np.nanpercentile(pred_ens, 25, axis=0) # [7,]
    
    iqr_daily = q75 - q25
    
    
    return float(np.nanmean(iqr_daily))


def calc_reliability_90pct(obs: np.ndarray, pred_ens: np.ndarray) -> float:
   
    if np.isnan(pred_ens).all():
        return np.nan
        
   
    q95 = np.nanpercentile(pred_ens, 95, axis=0) # [7,]
    q05 = np.nanpercentile(pred_ens, 5, axis=0) # [7,]
    
   
    mask = ~np.isnan(obs) & ~np.isnan(q95) & ~np.isnan(q05)
    if mask.sum() == 0:
        return np.nan
        
    obs_clean, q95_clean, q05_clean = obs[mask], q95[mask], q05[mask]
    
    
    is_covered = (obs_clean >= q05_clean) & (obs_clean <= q95_clean)
    
   
    return float(np.mean(is_covered))

def calc_sharpness_90pct(pred_ens: np.ndarray) -> float:
  
    
    if np.isnan(pred_ens).all(): return np.nan
    q95 = np.nanpercentile(pred_ens, 95, axis=0) # [7,]
    q05 = np.nanpercentile(pred_ens, 5, axis=0) # [7,]
    width_daily = q95 - q05
    return float(np.nanmean(width_daily))

In [ ]:
# Figure 4 and Figure 9  NSE and KGE 

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path


CSV_HYDRO = Path("fullseries_all_metrics_hydroforcast.csv")  # fullseries_hydroforecast_genelization for figure 9
CSV_BENCH = Path("fullseries_all_metrics_ALD.csv")   #fullseries_ALD_genelization  for  figure 9

USE_LEAD = "1"

plt.style.use('default')

def filter_lead(df, use_lead="1"):
    
    if "lead" in df.columns:
        if df["lead"].dtype.kind in "iuf":  
            return df[df["lead"] == (int(use_lead) if use_lead.isdigit() else use_lead)].copy()
        else:
            return df[df["lead"].astype(str) == str(use_lead)].copy()
    return df.copy()

def get_clean_data(df, col_name):
    
    series = df.get(col_name, pd.Series())
    x = pd.to_numeric(series, errors="coerce").to_numpy(float)
    x = x[np.isfinite(x)]  
    x = x[(x >= -1) & (x <= 1)] 
    return x

def calculate_ecdf(x):
    
    x = np.sort(x)
    y = np.arange(1, len(x)+1) / len(x) if len(x) else np.array([])
    return x, y


df_hydro_raw = pd.read_csv(CSV_HYDRO)
df_bench_raw = pd.read_csv(CSV_BENCH)

df_hydro = filter_lead(df_hydro_raw, USE_LEAD)
df_bench = filter_lead(df_bench_raw, USE_LEAD)


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False 


sns.set_theme(style="ticks", font_scale=1.1)


COLOR_HYDRO = "#117a65"  
COLOR_BENCH = "#d35400"  
LINEWIDTH = 2.0            


fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)

metrics = [
    {"col": "KGE", "title": "KGE", "ax_label": "(a)"},
    {"col": "NSE", "title": "NSE", "ax_label": "(b)"}
]

for i, metric in enumerate(metrics):
    ax = axes[i]
    col = metric["col"]
    
    
    x_hydro = get_clean_data(df_hydro, col)
    x_bench = get_clean_data(df_bench, col)
    
    
    hx, hy = calculate_ecdf(x_hydro)
    bx, by = calculate_ecdf(x_bench)
    
    
    med_h = np.median(x_hydro) if len(x_hydro) > 0 else np.nan
    med_b = np.median(x_bench) if len(x_bench) > 0 else np.nan
    
    
    label_hydro = f'HydroForecast (median={med_h:.2f})'
    label_bench = f'Benchmark (median={med_b:.2f})'
    
    
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, color='grey', alpha=0.3, zorder=0)
    
    
    ax.plot(hx, hy, lw=LINEWIDTH, color=COLOR_HYDRO, label=label_hydro, zorder=10)
   
    ax.plot(bx, by, lw=LINEWIDTH, color=COLOR_BENCH, linestyle='--', label=label_bench, zorder=10)
    
    
    ax.set_title(metric["title"], fontweight='bold', pad=12)
    ax.set_xlabel(f"{col} Score")
    ax.set_xlim(-1.0, 1.02) 
    ax.set_ylim(0, 1.02)
    
   
    if i == 0:
        ax.set_ylabel("ECDF")
    
    
    ax.legend(loc='upper left', frameon=False, fontsize=10, handlelength=2.5)


sns.despine() 
plt.tight_layout()



plt.show()

In [ ]:
# figure 5 and figure 9 crps , coverage 

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
CSV_HYDRO = Path("fullseries_hydroforcast.csv") 
CSV_BENCH = Path("fullseries_benchmark.csv")

USE_LEAD = "1"
SAVE_PATH = '' 
plt.style.use('default')

def filter_lead(df, use_lead="1"):
    if "lead" in df.columns:
        if df["lead"].dtype.kind in "iuf":
            return df[df["lead"] == (int(use_lead) if use_lead.isdigit() else use_lead)].copy()
        else:
            return df[df["lead"].astype(str) == str(use_lead)].copy()
    return df.copy()

def get_clean_data(df, col_name, limit_range=None):
    series = df.get(col_name, pd.Series())
    x = pd.to_numeric(series, errors="coerce").to_numpy(float)
    x = x[np.isfinite(x)]
    if limit_range:
        x = x[(x >= limit_range[0]) & (x <= limit_range[1])]
    return x


df_hydro_raw = pd.read_csv(CSV_HYDRO)
df_bench_raw = pd.read_csv(CSV_BENCH)

df_hydro = filter_lead(df_hydro_raw, USE_LEAD)
df_bench = filter_lead(df_bench_raw, USE_LEAD)


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="ticks", font_scale=1.1)

COLOR_HYDRO = "#117a65"  
COLOR_BENCH = "#d35400"  

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# ==========================================
# Plot 1 (Left): CRPS (Mean)
# ==========================================
ax = axes[0]
col = "CRPS_mean"


LIMIT_X = 2.0 
h_data = get_clean_data(df_hydro, col)
b_data = get_clean_data(df_bench, col)

med_h = np.median(h_data) if len(h_data) > 0 else 0
med_b = np.median(b_data) if len(b_data) > 0 else 0

bins = np.linspace(0, LIMIT_X, 30)


ax.hist(h_data, bins=bins, density=True, histtype='step', lw=2.5, 
        color=COLOR_HYDRO, label=f'HydroForecast (med={med_h:.2f})', zorder=10)
ax.hist(b_data, bins=bins, density=True, histtype='step', lw=2.5, 
        color=COLOR_BENCH, linestyle='--', label=f'Benchmark (med={med_b:.2f})', zorder=9)


ax.hist(h_data, bins=bins, density=True, histtype='stepfilled', alpha=0.1, color=COLOR_HYDRO)
ax.hist(b_data, bins=bins, density=True, histtype='stepfilled', alpha=0.05, color=COLOR_BENCH)

ax.set_xlabel("CRPS Value (Lower is better)")
ax.set_ylabel("Probability Density")
ax.set_xlim(0, LIMIT_X)


ax.legend(loc='upper left', bbox_to_anchor=(0.01, 1.1),frameon=False, fontsize=9.5)


# ==========================================
# Plot 2 (Right): Reliability (90% Coverage)
# ==========================================
ax = axes[1]
col = "Reliability_90pct"

h_data = get_clean_data(df_hydro, col, limit_range=(0, 1))
b_data = get_clean_data(df_bench, col, limit_range=(0, 1))

med_h = np.median(h_data) if len(h_data) > 0 else 0
med_b = np.median(b_data) if len(b_data) > 0 else 0

bins = np.linspace(0, 1, 26)


ax.hist(h_data, bins=bins, density=True, histtype='step', lw=2.5, 
        color=COLOR_HYDRO, label=f'HydroForecast', zorder=10)
ax.hist(b_data, bins=bins, density=True, histtype='step', lw=2.5, 
        color=COLOR_BENCH, linestyle='--', label=f'Benchmark', zorder=9)


ax.hist(h_data, bins=bins, density=True, histtype='stepfilled', alpha=0.1, color=COLOR_HYDRO)
ax.hist(b_data, bins=bins, density=True, histtype='stepfilled', alpha=0.05, color=COLOR_BENCH)


ax.axvline(0.9, color='gray', linestyle=':', lw=1.5, label='Ideal (0.9)')



ax.set_xlabel("Coverage Probability")

ax.set_xlim(0, 1)


ax.legend(loc='upper left', bbox_to_anchor=(0.01, 1.1), frameon=False, fontsize=9.5)


for ax in axes:
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, color='grey', alpha=0.3, zorder=0)

sns.despine()
plt.tight_layout()

plt.show()

In [ ]:
# Figure 5
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path


CSV_HYDRO = Path("fullseries_hydroforcast.csv") 
CSV_BENCH = Path("fullseries_benchmark.csv")
CSV_GEO   = Path("geo_statics.csv")

USE_LEAD = "1" 



COLOR_BETTER = "#005fafff" 
COLOR_WORSE  = "#d62728ff" 


def clean_station_id(series):
    
    return series.astype(str).str.replace('GRDC_', '', regex=False).str.strip()

def load_and_filter(csv_path, lead):
    
    df = pd.read_csv(csv_path)
    
    
    df['station_clean'] = clean_station_id(df['station'])
    
    
    if "lead" in df.columns:
        
        if lead in df['lead'].values:
            df = df[df["lead"] == lead].copy()
            
        else:
           
            unique_leads = df['lead'].unique()
            
    
    return df



df_h = load_and_filter(CSV_HYDRO, USE_LEAD)
df_b = load_and_filter(CSV_BENCH, USE_LEAD)

if len(df_h) == 0 or len(df_b) == 0:
    
    exit()


df_geo = pd.read_csv(CSV_GEO)

geo_station_col = next((c for c in ['stations', 'station', 'station_id', 'no'] if c in df_geo.columns), None)
if not geo_station_col:
    raise ValueError("can not find station ")

df_geo['station_clean'] = clean_station_id(df_geo[geo_station_col])


cols = ['station_clean', 'CRPS_mean']

df_perf = pd.merge(
    df_h[cols].rename(columns={'CRPS_mean': 'CRPS_Hydro'}),
    df_b[cols].rename(columns={'CRPS_mean': 'CRPS_Bench'}),
    on='station_clean', how='inner'
)


df_perf['pct_change'] = (df_perf['CRPS_Hydro'] - df_perf['CRPS_Bench']) / df_perf['CRPS_Hydro'] * 100


df_final = pd.merge(df_perf, df_geo[['station_clean', 'long', 'lat']], on='station_clean', how='inner')
df_final = df_final.dropna(subset=['long', 'lat', 'pct_change'])




plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())


ax.set_global()


ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='#444444', zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.15, color='#666666', zorder=1)
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', zorder=0) 
ax.add_feature(cfeature.OCEAN, facecolor='#eef8ff', zorder=0)
ax.add_feature(cfeature.COASTLINE,linewidth=0.6)

sc = ax.scatter(
    df_final['long'], df_final['lat'],
    c=df_final['pct_change'],
    s=20, 
    cmap='RdBu_r', 
    transform=ccrs.PlateCarree(),
    alpha=0.9,
    edgecolor='black', 
    linewidth=0.1,
    vmin=-50, 
    vmax=50,
    zorder=10 # 
)


ax.set_title("Relative Improvement in CRPS (Global Distribution)", fontweight='bold', fontsize=14, pad=15)
fig.text(0.13, 0.88, '(a)', fontsize=16, fontweight='bold', ha='left')


cax = fig.add_axes([0.25, 0.05, 0.5, 0.03])
cb = plt.colorbar(sc, cax=cax, orientation='horizontal', extend='both')
cb.set_label('CRPS Percentage Change [%]', fontsize=10, labelpad=5)
cb.ax.tick_params(labelsize=9)




plt.show()



In [ ]:
# Figure 6

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns


CSV_BASE = Path("/home/huangbh/eval_results_11-6/floodonly_benchmark.csv")
CSV_CFG = Path("/home/huangbh/eval_results_11-6/floodonly_hydroforecast.csv")


COLOR_HYDRO = "#005fafff"  # (HydroForecast )
COLOR_BASE  = "#d62728ff"  #  (Baseline)


df_base = pd.read_csv(CSV_BASE)
df_hydro = pd.read_csv(CSV_CFG)


metrics = [
   
    ("PBIAS_peak", "Peak Bias", "Peak Bias (%)", 0, (-100, 100)),
    ("PBIAS_vol",  "Volume Bias", "Volume Bias (%)", 0, (-100, 100)),
    ("Peak_Timing_Error", "Peak Timing Error", "Timing Error (days)", 0, (-7.5, 7.5)),
    ("r_shape", "Hydrograph Shape", "Shape Coefficient (-)", 1, (-0.5, 1.05)) 
]


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="ticks", font_scale=1.1) 

fig, axes = plt.subplots(2, 2, figsize=(12, 10)) 
axes = axes.flatten()


labels = ['(a)', '(b)', '(c)', '(d)']

for i, (col, title, xlabel, ref_val, x_limit) in enumerate(metrics):
    ax = axes[i]
    
   
    d1 = pd.to_numeric(df_base[col], errors='coerce').dropna().values
    d2 = pd.to_numeric(df_hydro[col], errors='coerce').dropna().values
    
    
    d1 = d1[np.isfinite(d1)]
    d2 = d2[np.isfinite(d2)]
    
    
    if col == "Peak_Timing_Error":
        
        bins = np.arange(-7.5, 8.5, 1)
        
        d1 = d1[(d1 >= -7) & (d1 <= 7)]
        d2 = d2[(d2 >= -7) & (d2 <= 7)]
    elif col == "r_shape":
        
        bins = np.linspace(-0.5, 1.0, 50)
        d1 = d1[(d1 >= -1) & (d1 <= 1)] 
        d2 = d2[(d2 >= -1) & (d2 <= 1)]
    else:
        
        combined = np.concatenate([d1, d2])
       
        d1 = d1[(d1 >= -100) & (d1 <= 100)]
        d2 = d2[(d2 >= -100) & (d2 <= 100)]
        bins = np.linspace(-100, 100, 41) 

    
    med1 = np.median(d1) if len(d1) > 0 else np.nan
    med2 = np.median(d2) if len(d2) > 0 else np.nan
    
   
    ax.hist(d2, bins=bins, density=False, histtype='step', lw=2.5, 
            color=COLOR_HYDRO, label=f'HydroForecast (med={med2:.2f})', zorder=10)
    
   
    ax.hist(d1, bins=bins, density=False, histtype='step', lw=2.5, 
            color=COLOR_BASE, linestyle='--', label=f'Baseline (med={med1:.2f})', zorder=9)
    
   
    ax.hist(d2, bins=bins, density=False, histtype='stepfilled', alpha=0.05, color=COLOR_HYDRO, zorder=1)
    ax.hist(d1, bins=bins, density=False, histtype='stepfilled', alpha=0.03, color=COLOR_BASE, zorder=0)

  
    ax.axvline(ref_val, color='#444444', linestyle=':', linewidth=1.5, alpha=0.8, label='Ideal')

    
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Event Count")
    
    
    if x_limit:
        ax.set_xlim(x_limit)
        
    
    ax.grid(True, which='major', linestyle='--', linewidth=0.7, color='grey', alpha=0.3)
    
   
    ax.text(-0.12, 1.05, labels[i], transform=ax.transAxes, 
            fontsize=14, fontweight='bold', va='top', ha='right')
    
    
    if col == "r_shape":
        ax.legend(loc='upper left', frameon=False, fontsize=10) 
    else:
        ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.0),frameon=False, fontsize=10)

sns.despine()

plt.tight_layout()

plt.show()

In [ ]:
# figure 8

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path


BASE_DIR = Path("T2")  # fold  T2/that per_station_event_metrics_lead1.csv and  T2/per_station_event_metrics_lead1_ALD.csv in 
CSV_GEO  = Path("geo_statics.csv")


USE_LEAD = 1  



COLOR_BETTER = "#005fafff"
COLOR_WORSE  = "#d62728ff" 


def load_specific_lead(base_dir, lead, is_benchmark=False):
    
    suffix = "_ALD.csv" if is_benchmark else ".csv"
    filename = f"per_station_event_metrics_lead{lead}_gap0_1.0{suffix}"
    path = base_dir / filename
    
    print(f"reading: {path.name}")
    if not path.exists():
        raise FileNotFoundError(f"no exist: {path}")
        
    df = pd.read_csv(path)
    
    
    st_col = next((c for c in ['station', 'stations', 'station_id', 'Unnamed: 0'] if c in df.columns), None)
    
    
    if 'station' in df.columns:
        df['station_clean'] = df['station'].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    elif 'Unnamed: 0' in df.columns: 
        df['station_clean'] = df['Unnamed: 0'].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    else:
        
        df['station_clean'] = df.iloc[:, 0].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
        
    return df




df_h = load_specific_lead(BASE_DIR, USE_LEAD, is_benchmark=False)
df_b = load_specific_lead(BASE_DIR, USE_LEAD, is_benchmark=True)


df_geo = pd.read_csv(CSV_GEO)
geo_st_col = next((c for c in ['stations', 'station', 'station_id', 'no'] if c in df_geo.columns), None)
df_geo['station_clean'] = df_geo[geo_st_col].astype(str).str.replace('GRDC_', '', regex=False).str.strip()


cols = ['station_clean', 'F1_Score']
df_perf = pd.merge(
    df_h[cols].rename(columns={'F1_Score': 'F1_Hydro'}),
    df_b[cols].rename(columns={'F1_Score': 'F1_Bench'}),
    on='station_clean', how='inner'
)


df_perf['f1_diff'] = df_perf['F1_Hydro'] - df_perf['F1_Bench']


df_final = pd.merge(df_perf, df_geo[['station_clean', 'long', 'lat']], on='station_clean', how='inner')
df_final = df_final.dropna(subset=['long', 'lat', 'f1_diff'])




plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

fig = plt.figure(figsize=(12, 6.5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())


ax.set_global() 


ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='#444444', zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.15, color='#666666', zorder=1)
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0', zorder=0) 
ax.add_feature(cfeature.OCEAN, facecolor='#eef8ff', zorder=0)
ax.add_feature(cfeature.COASTLINE,linewidth=0.6)

V_LIMIT = 0.3 


sc = ax.scatter(
    df_final['long'], df_final['lat'],
    c=df_final['f1_diff'],
    s=18, 
    cmap='RdBu_r', 
    transform=ccrs.PlateCarree(),
    alpha=0.9,
    edgecolor='black', 
    linewidth=0.1,
    vmin=-V_LIMIT, 
    vmax=V_LIMIT,
    zorder=10 
)


ax.set_title(f"Difference in F1 Score (HydroForecast - Benchmark) [Lead {USE_LEAD}]", fontweight='bold', fontsize=14, pad=15)
fig.text(0.13, 0.88, '(b)', fontsize=16, fontweight='bold', ha='left') 


cax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
cb = plt.colorbar(sc, cax=cax, orientation='horizontal', extend='both')
cb.set_label('F1 Score Difference [-]', fontsize=10, labelpad=5)
cb.ax.tick_params(labelsize=9)





plt.show()

In [ ]:
# figure 7
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


BASE_DIR = Path("T2/")  #T5


COLOR_HYDRO = "#005fafff"  #
COLOR_BENCH = "#d62728ff"  # 




SHOW_TREND_LINE = False 


def load_data(base_dir, leads=range(1, 8)):
    data_list = []
    for lead in leads:
        p_hydro = base_dir / f"per_station_event_metrics_lead{lead}.csv"
        p_bench = base_dir / f"per_station_event_metrics_lead{lead}_ALD.csv"
        
        if p_hydro.exists():
            df = pd.read_csv(p_hydro)
            if 'Precision' in df.columns:
                vals = pd.to_numeric(df['Precision'], errors='coerce').dropna()
                for v in vals: data_list.append({'Lead': lead, 'Model': 'HydroForecast', 'Precision': v})

        if p_bench.exists():
            df = pd.read_csv(p_bench)
            if 'Precision' in df.columns:
                vals = pd.to_numeric(df['Precision'], errors='coerce').dropna()
                for v in vals: data_list.append({'Lead': lead, 'Model': 'Benchmark', 'Precision': v})
            
    return pd.DataFrame(data_list)


print("Loading data...")
df_all = load_data(BASE_DIR)
print(f"Loaded {len(df_all)} records.")

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="white", font_scale=1.2)

fig, ax = plt.subplots(figsize=(10, 6))


box = sns.boxplot(
    data=df_all, 
    x='Lead', 
    y='Precision', 
    hue='Model',
    palette=[COLOR_HYDRO, COLOR_BENCH],
    width=0.6,       
    linewidth=1.2,
    fliersize=0.5,
    ax=ax,
    saturation=1.0,
    legend=False
)


for patch in ax.patches:
    col = patch.get_facecolor()
    patch.set_facecolor((col[0], col[1], col[2], 0.2)) 
    patch.set_edgecolor((col[0], col[1], col[2], 1.0)) 


if SHOW_TREND_LINE:
    n_lines_before = len(ax.lines)
    sns.pointplot(
        data=df_all, 
        x='Lead', 
        y='Precision', 
        hue='Model', 
        estimator=np.median, 
        errorbar=None,
        palette=[COLOR_HYDRO, COLOR_BENCH],
        markers='o',     
        linestyles='--',  
        scale=0.6,        
        dodge=0.4,        
        ax=ax,
        legend=False
    )
    
    new_lines = ax.lines[n_lines_before:]
    for line in new_lines:
        line.set_alpha(0.6) 
        line.set_zorder(10) 




ax.set_ylim(-0.05, 1.05)


ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])


ax.yaxis.grid(True, linestyle='--', which='major', color='grey', alpha=0.3)
ax.xaxis.grid(False)


ax.set_xlabel("Forecast Lead Time (days)", fontweight='bold', labelpad=10)
ax.set_ylabel("Precision", fontweight='bold', labelpad=10)
ax.set_title("Precision Distribution (2-Year Return Period)", fontweight='bold', pad=15)


sns.despine(left=True) 

plt.tight_layout()

plt.show()

In [ ]:
# figure 10
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


METRIC_CSV = Path("fullseries_hydroforecast_genelization.csv")
GEO_CSV    = Path("geo_statics.csv")




RESERVOIR_AREA_COL = 'res_area_GeoDAR_v11_reservoirs_km2'


PALETTE = {"Natural": "#005fafff", "Regulated": "#d62728ff"}


def load_data():
    print("Loading Data...")
    if not METRIC_CSV.exists(): return pd.DataFrame()
    
   
    df_m = pd.read_csv(METRIC_CSV)
    st_col_m = next((c for c in ['stations', 'station'] if c in df_m.columns), df_m.columns[0])
    df_m['sid'] = df_m[st_col_m].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    

    df_g = pd.read_csv(GEO_CSV)
    st_col_g = next((c for c in ['stations', 'station'] if c in df_g.columns), df_g.columns[0])
    df_g['sid'] = df_g[st_col_g].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    
   
    cols_geo = ['sid', RESERVOIR_AREA_COL]
    df = pd.merge(df_m[['sid', 'NSE', 'Reliability_90pct']], df_g[cols_geo], on='sid', how='inner')
    
    
    df[RESERVOIR_AREA_COL] = df[RESERVOIR_AREA_COL].fillna(0)
    
    
    df['Basin Type'] = np.where(df[RESERVOIR_AREA_COL] > 0, 'Regulated', 'Natural')
    
    return df


df = load_data()
if df.empty: exit()


n_counts = df['Basin Type'].value_counts()
n_nat = n_counts.get('Natural', 0)
n_reg = n_counts.get('Regulated', 0)

print(f"Sample Sizes: Natural={n_nat}, Regulated={n_reg}")

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="white", font_scale=1.1)

fig, axes = plt.subplots(1, 2, figsize=(10, 6))


ORDER = ['Natural', 'Regulated']


ax_nse = axes[0]
df_nse = df[df['NSE'] > -1].copy()

sns.boxplot(
    data=df_nse,
    x='Basin Type',
    y='NSE',
    order=ORDER,
    palette=PALETTE,
    width=0.5,
    linewidth=1.5,
    fliersize=1.5, 
    ax=ax_nse,
    saturation=1.0
)

ax_nse.set_title("NSE Score Comparison", fontweight='bold', pad=15)
ax_nse.set_ylabel("NSE Score")
ax_nse.set_ylim(-1.05, 1.05)
ax_nse.axhline(0, color='grey', linestyle='--', linewidth=1, alpha=0.5)

# --- Plot 2: Reliability ---
ax_rel = axes[1]
df_rel = df.dropna(subset=['Reliability_90pct']).copy()

sns.boxplot(
    data=df_rel,
    x='Basin Type',
    y='Reliability_90pct',
    order=ORDER,
    palette=PALETTE,
    width=0.5,
    linewidth=1.5,
    fliersize=1.5,
    ax=ax_rel,
    saturation=1.0
)

ax_rel.set_title("Reliability Comparison", fontweight='bold', pad=15)
ax_rel.set_ylabel("Coverage Ratio")
ax_rel.set_ylim(0.0, 1.05)
ax_rel.axhline(0.9, color='#444444', linestyle=':', linewidth=1.5, label='Ideal')
ax_rel.legend(loc='lower right', frameon=False, fontsize=10)


for ax in axes:
    
    for patch in ax.patches:
        r, g, b, a = patch.get_facecolor()
        patch.set_facecolor((r, g, b, 0.3)) 
        patch.set_edgecolor((r, g, b, 1.0)) 

    
    labels = [f"Natural\n(N={n_nat})", f"Regulated\n(N={n_reg})"]
    ax.set_xticklabels(labels, fontweight='bold')
    ax.set_xlabel("") 

    
    ax.yaxis.grid(True, linestyle='--', color='grey', alpha=0.3)
    ax.xaxis.grid(False)


axes[0].text(-0.15, 1.05, '(a)', transform=axes[0].transAxes, fontsize=16, fontweight='bold')
axes[1].text(-0.15, 1.05, '(b)', transform=axes[1].transAxes, fontsize=16, fontweight='bold')

sns.despine(left=True)
plt.tight_layout()

print(f"Saving figure to {SAVE_PATH}...")
plt.savefig(SAVE_PATH, dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# figure 11
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path


CSV_KOPPEN = Path("/koppen_classification.csv")
CSV_HYDRO  = Path("fullseries_hydroforecast_genelization.csv")



TARGET_CLASSES = ['Af', 'Cfa', 'Cfb', 'Csa', 'Csb', 'Dfb', 'Dfc', 'Dwb']


ZONE_PALETTE = {
    'Af':  '#d62728', # Red
    'Cfa': '#2ca02c', # Green
    'Cfb': '#98df8a', # Light Green
    'Csa': '#8c564b', # Brown
    'Csb': '#e377c2', # Pink
    'Dfb': '#1f77b4', # Blue
    'Dfc': '#aec7e8', # Light Blue
    'Dwb': '#9467bd', # Purple
}


def load_data():
    print("Loading Data...")
    if not CSV_HYDRO.exists(): return pd.DataFrame()
    
    # Koppen
    df_k = pd.read_csv(CSV_KOPPEN)
    st_col = next((c for c in ['stations', 'station'] if c in df_k.columns), df_k.columns[0])
    df_k['sid'] = df_k[st_col].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    station_to_cls = df_k.set_index('sid')['secondary_class'].to_dict()
    
    # Metrics
    df = pd.read_csv(CSV_HYDRO)
    st_col_h = next((c for c in ['station', 'stations'] if c in df.columns), df.columns[0])
    df['sid'] = df[st_col_h].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    df['Subclass'] = df['sid'].map(station_to_cls)
    
    return df[df['Subclass'].isin(TARGET_CLASSES)].copy()


df = load_data()
if df.empty: exit()

# 排序
df['Subclass'] = pd.Categorical(df['Subclass'], categories=TARGET_CLASSES, ordered=True)
df.sort_values('Subclass', inplace=True)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="ticks", font_scale=1.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot 1: NSE Boxplot ---
ax_nse = axes[0]
df_nse = df[df['NSE'] > -1].copy()


sns.boxplot(
    data=df_nse, 
    x='Subclass', 
    y='NSE', 
    palette=ZONE_PALETTE, 
    width=0.6, 
    linewidth=1.5,      
    fliersize=1.5,      
    ax=ax_nse,
    saturation=1.0     
)


ax_nse.set_title("NSE Distribution by Climate Subclass", fontweight='bold', pad=15)
ax_nse.set_ylabel("NSE Score")
ax_nse.set_xlabel("")
ax_nse.set_ylim(-1.05, 1.05)
ax_nse.axhline(0, color='grey', linestyle='--', linewidth=1, alpha=0.5)


ax_rel = axes[1]
df_rel = df.dropna(subset=['Reliability_90pct']).copy()

sns.boxplot(
    data=df_rel, 
    x='Subclass', 
    y='Reliability_90pct', 
    palette=ZONE_PALETTE, 
    width=0.6, 
    linewidth=1.5, 
    fliersize=1.5, 
    ax=ax_rel,
    saturation=1.0
)


ax_rel.set_title("Reliability (Target: 0.90)", fontweight='bold', pad=15)
ax_rel.set_ylabel("Coverage Ratio")
ax_rel.set_xlabel("")
ax_rel.set_ylim(0.0, 1.05)

ax_rel.axhline(0.9, color='#444444', linestyle=':', linewidth=1.5)


for ax in axes:
    
    
    for patch in ax.patches:
        r, g, b, a = patch.get_facecolor()
        patch.set_facecolor((r, g, b, 0.3)) 
        
     
        patch.set_edgecolor((r, g, b, 1.0)) 
        

    
    counts = df.groupby('Subclass')['sid'].nunique()
    new_labels = []
    current_labels = [label.get_text() for label in ax.get_xticklabels()]
    for label in current_labels:
        n = counts.get(label, 0)
        new_labels.append(f"{label}\n(N={n})")
    ax.set_xticklabels(new_labels)

    
    ax.yaxis.grid(True, linestyle='--', color='grey', alpha=0.3)
    ax.xaxis.grid(False)


axes[0].text(-0.1, 1.05, '(a)', transform=axes[0].transAxes, fontsize=16, fontweight='bold')
axes[1].text(-0.1, 1.05, '(b)', transform=axes[1].transAxes, fontsize=16, fontweight='bold')

sns.despine(left=True) 
plt.tight_layout()


plt.show()

In [ ]:
# figure 11
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path


CSV_KOPPEN = Path("/koppen_classification.csv")
CSV_METRICS = Path("fullseries_hydroforecast_genelization.csv")
CSV_GEO    = Path("geo_statics.csv")



TARGET_CLASSES = ['Af', 'Cfa', 'Cfb', 'Csa', 'Csb', 'Dfb', 'Dfc', 'Dwb']




SIZE_MIN = 10    
SIZE_MAX = 120   


def load_and_merge():
    print("Loading Data...")
    
    df_k = pd.read_csv(CSV_KOPPEN)
    st_col = next((c for c in ['stations', 'station'] if c in df_k.columns), df_k.columns[0])
    df_k['sid'] = df_k[st_col].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    df_k = df_k[df_k['secondary_class'].isin(TARGET_CLASSES)].copy()
    
  
    if not CSV_METRICS.exists(): return pd.DataFrame()
    df_m = pd.read_csv(CSV_METRICS)
    st_col_m = next((c for c in ['stations', 'station'] if c in df_m.columns), df_m.columns[0])
    df_m['sid'] = df_m[st_col_m].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    
    
    df_g = pd.read_csv(CSV_GEO)
    st_col_g = next((c for c in ['stations', 'station'] if c in df_g.columns), df_g.columns[0])
    df_g['sid'] = df_g[st_col_g].astype(str).str.replace('GRDC_', '', regex=False).str.strip()
    
    
    df = pd.merge(df_m[['sid', 'NSE']], df_k[['sid', 'secondary_class']], on='sid', how='inner')
    df = pd.merge(df, df_g[['sid', 'long', 'lat']], on='sid', how='inner')
    
    return df

def calculate_size(nse_series):
    val = nse_series.clip(lower=0)
    
    return SIZE_MIN + (val * (SIZE_MAX - SIZE_MIN))


df = load_and_merge()
if df.empty: exit()

df['point_size'] = calculate_size(df['NSE'])


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.set_global()


ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color='#444444', zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.15, color='#666666', zorder=1)
ax.add_feature(cfeature.LAND, facecolor='#f4f4f4', zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='#eef8ff', zorder=0)
ax.add_feature(cfeature.COASTLINE,linewidth=0.6)


legend_handles = []
for zone in TARGET_CLASSES:
    subset = df[df['secondary_class'] == zone]
    if subset.empty: continue
    
    sc = ax.scatter(
        subset['long'], subset['lat'],
        s=subset['point_size'],
        c=[ZONE_PALETTE[zone]], 
        transform=ccrs.PlateCarree(),
        alpha=0.9,
        edgecolor='white', 
        linewidth=0.4,     
        zorder=10,
        label=zone
    )
    legend_handles.append(sc)


ax.set_title("Global Performance (NSE) Distribution by Climate Zone", fontweight='bold', fontsize=16, pad=20)




sizes_to_show = [0.0, 0.5, 1.0]
size_handles = []
for nse_val in sizes_to_show:
    s = SIZE_MIN + (nse_val * (SIZE_MAX - SIZE_MIN))
    handle = ax.scatter([], [], s=s, c='gray', edgecolors='none', label=f'NSE = {nse_val}')
    size_handles.append(handle)




plt.show()